# 04 — Label-blind reserve-list recovery checkpoint
The checkpoint examines the same corrupted top-3 list and the precomputed reserve ranking. It restores the highest-ranked clean candidate missing from the corrupted top-3, replacing the injected item. It does not inspect gold labels or evidence annotations. This is a fixed retrieval recovery rule, not an adaptive human checkpoint.

In [ ]:
from pathlib import Path
import json
BASE=Path("/content/scifact_study")
def read_jsonl(p):
    with open(p,encoding="utf-8") as f: return [json.loads(x) for x in f if x.strip()]
corrupted=read_jsonl(BASE/"results/corrupted_rankings.jsonl")
recovered=[]
for r in corrupted:
    current=r["doc_ids"].copy()
    reserve=r["reserve_doc_ids"]
    replacement=next((d for d in reserve if d not in current),None)
    if replacement is None:
        raise RuntimeError(f"No reserve candidate for claim {r['claim_id']}")
    # Replace the injected first slot, preserve the other two retrieved items.
    recovered_ids=[replacement]+current[1:]
    recovered.append({
        "claim_id":r["claim_id"],
        "doc_ids":recovered_ids,
        "recovered_doc_id":replacement,
        "removed_injected_doc_id":r["injected_doc_id"]
    })
print("Recovered claims:",len(recovered))
print("Before:",corrupted[0]["doc_ids"])
print("After: ",recovered[0]["doc_ids"])


In [ ]:
out=BASE/"results"
with open(out/"recovered_rankings.jsonl","w",encoding="utf-8") as f:
    for r in recovered: f.write(json.dumps(r)+"\n")
print("Saved recovered rankings.")


## Fair-comparison note
The checkpoint has access to a precomputed reserve list, so it has extra retrieval information. Report the intervention as reserve-list recovery and include candidate counts/runtime. Do not describe it as compute-matched unless a matched-budget control is implemented.